In [1]:
import pandas as pd
import kaggle_benchmarks as kbench
import os
import time
import re

# 1. Keep the Kaggle working directory clean
os.environ["RENDER_SUBRUNS"] = "False"

# 2. Load the dataset
dataset_path = "/kaggle/input/datasets/ashishpatnaik2806/corrupted-gol-benchmark/corrupted_gol_benchmark.csv"
df = pd.read_csv(dataset_path)
print(f"Loaded {df.shape[0]} tasks for evaluation.")

# 3. BEST-IN-CLASS list extractor (merged from two approaches)
def extract_list(text):
    text_str = str(text)

    # PRIMARY: Match actual coordinate tuple lists ONLY.
    # This pattern only matches [] or [(digit,digit),...] format.
    # Correctly skips chatty brackets like [5x5], [the rules], [1], [Step 2] etc.
    tuple_match = re.search(
        r'\[\s*(?:\(\s*\d+\s*,\s*\d+\s*\)\s*,?\s*)*\]',
        text_str
    )
    if tuple_match:
        return tuple_match.group(0)

    # FALLBACK: Find all non-nested bracket groups, return the longest.
    # Uses [^\[\]]* (no nesting) instead of .*? (non-greedy) to avoid
    # the bug where "Based on [X], answer: [Y]" incorrectly returns [X].
    brackets = re.findall(r'\[[^\[\]]*\]', text_str, re.DOTALL)
    if brackets:
        return max(brackets, key=len)

    # LAST RESORT: return raw response
    return text_str


# 4. Task definition with smart parsing and retry
@kbench.task(store_task=False)
def evaluate_single_grid(llm, task_id, tier, mutation, prompt,
                         expected_output, **kwargs) -> dict:
    max_retries = 3
    response = ""
    for attempt in range(max_retries):
        try:
            response = llm.prompt(prompt)
            break
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"API glitch on task {task_id}. Retrying in 5 seconds...")
            time.sleep(5)

    extracted_response = extract_list(response)

    clean_expected  = str(expected_output).replace(" ", "").replace("\n", "")
    clean_predicted = extracted_response.replace(" ", "").replace("\n", "")
    is_correct = (clean_expected == clean_predicted)

    kbench.assertions.assert_true(
        is_correct,
        expectation=(f"Model failed {mutation}. "
                     f"Expected {clean_expected}, got {clean_predicted}")
    )

    return {
        "task_id":    task_id,
        "tier":       tier,
        "mutation":   mutation,
        "is_correct": is_correct,
    }


# 5. Aggregator with per-mutation and per-tier breakdowns
@kbench.task(name="corrupted_gol_ros_v3")
def run_full_benchmark(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = evaluate_single_grid.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            remove_run_files=True,
        )

    eval_df = runs.as_dataframe()

    print("\n── Per-Mutation Accuracy ──────────────────────")
    mutation_acc = (
        eval_df.assign(is_correct=eval_df.result.str.get("is_correct"))
               .groupby(eval_df.result.str.get("mutation"))["is_correct"]
               .mean()
               .sort_values(ascending=False)
    )
    for mutation, acc in mutation_acc.items():
        print(f"  {mutation}: {acc * 100:.1f}%")

    print("\n── Per-Tier Accuracy ──────────────────────────")
    tier_acc = (
        eval_df.assign(is_correct=eval_df.result.str.get("is_correct"))
               .groupby(eval_df.result.str.get("tier"))["is_correct"]
               .mean()
               .sort_values(ascending=False)
    )
    for tier, acc in tier_acc.items():
        print(f"  {tier}: {acc * 100:.1f}%")

    overall_accuracy = float(eval_df.result.str.get("is_correct").mean())
    return overall_accuracy


# 6. Multi-model execution
available_models = {
    "Gemini 2.5 Flash": kbench.llm,
    "Claude Sonnet 4.5": kbench.llms.get("anthropic/claude-sonnet-4-5@20250929"),
    "Qwen 3 235B":       kbench.llms.get("qwen/qwen3-235b-a22b-instruct-2507"),
}

models_to_test = {
    name: model
    for name, model in available_models.items()
    if model is not None
}

results = {}
for model_name, target_llm in models_to_test.items():
    print(f"\n\n==================================================")
    print(f"STARTING EVALUATION: {model_name}")
    print(f"==================================================")

    run_obj       = run_full_benchmark.run(llm=target_llm, df=df)
    overall_score = run_obj.result

    results[model_name] = overall_score
    print(f"\nFINAL ROS FOR {model_name}: {overall_score * 100:.2f}%")
    print(f"==================================================")

# Final leaderboard summary
print("\n\n══════════════════════════════════════════════════")
print("  FINAL LEADERBOARD — Rule Override Score (ROS)")
print("══════════════════════════════════════════════════")
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<25} {score * 100:.2f}%")
print("══════════════════════════════════════════════════")

Loaded 600 tasks for evaluation.


STARTING EVALUATION: Gemini 2.5 Flash


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:  4.7min


[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 12.3min


[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed: 22.2min


...

[Parallel(n_jobs=4)]: Done 384 tasks      | elapsed:  1.7min


[Parallel(n_jobs=4)]: Done 504 tasks      | elapsed:  2.1min


[Parallel(n_jobs=4)]: Done 600 out of 600 | elapsed:  2.4min finished



── Per-Mutation Accuracy ──────────────────────
  R7: 15.0%
  R3: 10.0%
  R9: 10.0%
  R8: 5.0%
  R10: 3.3%
  R1: 3.3%
  R6: 3.3%
  R2: 1.7%
  R5: 0.0%
  R4: 0.0%

── Per-Tier Accuracy ──────────────────────────
  Pattern: 20.7%
  Easy: 0.0%
  Hard: 0.0%
  Medium: 0.0%



FINAL ROS FOR Qwen 3 235B: 5.17%


══════════════════════════════════════════════════
  FINAL LEADERBOARD — Rule Override Score (ROS)
══════════════════════════════════════════════════
  Gemini 2.5 Flash          31.00%
  Claude Sonnet 4.5         6.33%
  Qwen 3 235B               5.17%
══════════════════════════════════════════════════


In [2]:
%choose corrupted_gol_ros_v3

Kept: corrupted_gol_ros_v3-run_id_Run_3_qwen_qwen3-235b-a22b-instruct-2507.run.json
Kept: corrupted_gol_ros_v3.task.json
